In [1]:
from __future__ import annotations
import networkx as nx
import numpy as np

In [2]:
from abc import ABC, abstractmethod

# Decorators for graph characteristics

def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls


class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass

In [3]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass


def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

In degree of a node is the number of predecessors it has.



In [4]:
# cada vez que se requiera la lista de TFs
def get_parent_nodes(G: nx.DiGraph):
    """Get the parent nodes of a graph."""
    return [i for i, k_out in G.out_degree() if k_out > 0]

In [5]:
@return_distribution
@use_direction
@use_selfloops  
class In_Degree(_Property): # Hereda de la clase _Property
    """In degree of the graph.

    The in degree of each node is defined as the number of predecessors it has.

    Methods:
        compute: Compute the in connectivity of the graph.
        norm_biol: Normalize the in connectivity of the graph to the number of parents.
        norm_network: Normalize the in connectivity of the graph to the number of nodes.
    """

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no in-degree distribution.")

    def compute(self) -> np.array:
        """Compute the in-degree of the graph.

        Returns:
            nparray: Array with in-degrees of every node in graph.    
        """
        a = [x for a,x in self.G.in_degree()]
        self._raw_value = np.array(a)
        return self._raw_value

    @check_raw_value    # Decorator to check if raw value is None. If it is, raise an error.
    def norm_biol(self) -> float:
        """Normalize the nparray with the in-degrees of the graph to the number of parents"""
        n_parents = len(get_parent_nodes(self.G))
        try:
            return self._raw_value * (1 / n_parents)
        except ZeroDivisionError:
            raise NormalizationError("Division by zero (no parent nodes). Cannot normalize with this approach.")

    @check_raw_value
    def norm_network(self) -> float:
        """Normalize the nparray with the in-degrees of the graph to the number of nodes."""
        return self._raw_value * (1 / self._n_nodes)  

In [6]:
import pytest

# null graph
G = nx.DiGraph()
with pytest.raises(NullGraphError) as e_info:
    property = In_Degree(G)

# empty graph with nodes
G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))    # add nodes, no edges
# no edges
property = In_Degree(G)
assert property.compute().all() == 0 # no edges [0, 0, ..., 0]
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol().all() == 0 # no edges [0, 0, ..., 0] * (1 / 0) = [0, 0, ..., 0]
assert property.norm_network().all() == 0 # no edges [0, 0, ..., 0] * (1 / 10) = [0, 0, ..., 0]

# add edges
# complete directed graph with self loops
expected = np.array([10,10,10,10,10,10,10,10,10,10])
expected_1 = np.array([1.,1.,1.,1.,1.,1.,1.,1.,1.,1.])

G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = In_Degree(G)
assert np.array_equal(property.compute(), expected) # [10, 10, ..., 10]
assert np.array_equal(property.norm_network(), expected_1) #[10, 10, ..., 10] * (1 / 10) = [1., 1., ..., 1.]
assert np.array_equal(property.norm_biol(), expected_1) #[10, 10, ..., 10] * (1 / 10) = [1., 1., ..., 1.]

# only half of the nodes are parents and regulate every node in the graph
expected = np.array([5,5,5,5,5,5,5,5,5,5])
expected_1 = np.array([1.,1.,1.,1.,1.,1.,1.,1.,1.,1.])
expected_2 = np.array([0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5])

G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = In_Degree(G)
assert np.array_equal(property.compute(), expected) # [5, 5, ..., 5]
assert np.array_equal(property.norm_biol(), expected_1) # [5, 5, ..., 5] * (1/5) = [1., 1., ..., 1.]
assert np.array_equal(property.norm_network(), expected_2)# [5, 5, ..., 5] * (1/10) = [0.5, 0.5, ..., 0.5]


ModuleNotFoundError: No module named 'pytest'